<a href="https://colab.research.google.com/github/trongphuoc293-del/GPA/blob/main/gpa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import data_table

# 1. Bật tính năng bảng tương tác đẹp mắt của Google Colab
data_table.enable_dataframe_formatter()

# 2. Đọc dữ liệu từ file CSV với đường dẫn tuyệt đối
file_path = '/content/gpa.csv'

try:
    # Đọc file với dấu phân cách là ';'
    df = pd.read_csv(file_path, sep=';')

    # Xử lý làm sạch dữ liệu: Đảm bảo cột 'điểm gpa' là dạng số (tránh lỗi nếu có khoảng trắng hoặc chữ)
    df['điểm gpa'] = pd.to_numeric(df['điểm gpa'], errors='coerce')

    # Bỏ qua các dòng có điểm GPA bị trống (NaN) do lỗi dữ liệu nếu có
    df = df.dropna(subset=['điểm gpa'])

    # 3. Tạo một widget thanh trượt (slider) để lọc dữ liệu theo điểm GPA
    min_gpa_val = float(df['điểm gpa'].min())
    max_gpa_val = float(df['điểm gpa'].max())

    gpa_slider = widgets.FloatSlider(
        value=min_gpa_val,
        min=min_gpa_val,
        max=max_gpa_val,
        step=0.1,
        description='Lọc Min GPA:',
        continuous_update=False,
        layout={'width': '500px'}
    )

    # Nơi chứa kết quả hiển thị
    output_table = widgets.Output()

    # Hàm cập nhật bảng khi kéo thanh trượt
    def update_table(change):
        with output_table:
            clear_output(wait=True)
            # Lọc dữ liệu
            filtered_df = df[df['điểm gpa'] >= change.new]
            print(f"📌 Đang hiển thị sinh viên có GPA >= {change.new:.2f} (Tổng: {len(filtered_df)} dòng)")
            display(filtered_df)

    # Gắn sự kiện kéo thanh trượt với hàm cập nhật
    gpa_slider.observe(update_table, names='value')

    # Hiển thị mặc định lần đầu tiên
    with output_table:
        print(f"📌 Đang hiển thị tất cả dữ liệu (Tổng: {len(df)} dòng)")
        display(df)

    # 4. Gom thanh trượt và bảng lại thành một giao diện hoàn chỉnh
    ui = widgets.VBox([
        widgets.HTML("<h3>📊 Giao diện tra cứu dữ liệu Khảo sát học tập</h3>"),
        widgets.HTML("<p><i>Kéo thanh trượt bên dưới để lọc danh sách sinh viên theo mức điểm GPA tối thiểu.</i></p>"),
        gpa_slider,
        output_table
    ])

    display(ui)

except FileNotFoundError:
    print(f"❌ Lỗi: Không tìm thấy file tại đường dẫn '{file_path}'. Vui lòng kiểm tra lại xem file đã được tải lên đúng vị trí chưa nhé!")
except Exception as e:
    print(f"❌ Có lỗi xảy ra trong quá trình đọc file: {e}")

In [2]:
!pip install gradio -q

In [6]:
import gradio as gr
import pandas as pd

# 1. Đọc và làm sạch dữ liệu
file_path = '/content/gpa.csv'
try:
    df = pd.read_csv(file_path, sep=';')

    # Chuyển đổi các cột số liệu để tránh lỗi
    df['điểm gpa'] = pd.to_numeric(df['điểm gpa'], errors='coerce')
    df['bạn dành bao nhiêu thời gian cho 1 tuần để học'] = pd.to_numeric(df['bạn dành bao nhiêu thời gian cho 1 tuần để học'], errors='coerce')

    # Xóa các dòng lỗi không có GPA
    df = df.dropna(subset=['điểm gpa'])

except Exception as e:
    print(f"Lỗi đọc file: {e}. Đảm bảo file gpa.csv đã được upload vào /content/")
    df = pd.DataFrame() # Tạo df rỗng nếu lỗi

# 2. Hàm xử lý logic lọc dữ liệu
def filter_data(min_gpa, min_study_hours, part_time):
    if df.empty:
        return pd.DataFrame()

    filtered_df = df.copy()

    # Lọc theo GPA
    filtered_df = filtered_df[filtered_df['điểm gpa'] >= min_gpa]

    # Lọc theo số giờ học
    filtered_df = filtered_df[filtered_df['bạn dành bao nhiêu thời gian cho 1 tuần để học'] >= min_study_hours]

    # Lọc theo tình trạng đi làm thêm
    if part_time != "Tất cả":
        filtered_df = filtered_df[filtered_df['bạn có làm thêm part time không'].astype(str).str.strip() == part_time]

    return filtered_df

# 3. Thiết kế giao diện App bằng Gradio
with gr.Blocks() as app:
    gr.Markdown("# 🎓 Ứng dụng tra cứu Dữ liệu Khảo sát sinh viên")
    gr.Markdown("Tùy chỉnh các bộ lọc bên trái để xem danh sách sinh viên phù hợp ở bảng bên phải.")

    with gr.Row():
        # Cột bên trái: Các công cụ bộ lọc
        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ Bộ lọc dữ liệu")
            min_gpa_input = gr.Slider(minimum=0.0, maximum=4.0, step=0.1, value=0.0, label="Điểm GPA tối thiểu")
            min_study_input = gr.Slider(minimum=0, maximum=50, step=1, value=0, label="Giờ tự học tối thiểu (tuần)")
            part_time_input = gr.Radio(choices=["Tất cả", "Có", "Không"], value="Tất cả", label="Có làm part-time không?")

            # Nút bấm để kích hoạt lọc
            filter_btn = gr.Button("🔍 Lọc danh sách", variant="primary")

        # Cột bên phải: Hiển thị kết quả
        with gr.Column(scale=3):
            output_table = gr.Dataframe(value=df, label="Bảng kết quả", interactive=False)

    # Kết nối nút bấm với hàm xử lý
    filter_btn.click(
        fn=filter_data,
        inputs=[min_gpa_input, min_study_input, part_time_input],
        outputs=output_table
    )

# 4. Khởi chạy App
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://55cff743a70b5f8f16.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
import gradio as gr
import pandas as pd

# 1. Đọc và làm sạch dữ liệu
file_path = '/content/gpa.csv'
try:
    df = pd.read_csv(file_path, sep=';')
    df['điểm gpa'] = pd.to_numeric(df['điểm gpa'], errors='coerce')
    df = df.dropna(subset=['điểm gpa'])
except Exception as e:
    print(f"Lỗi đọc file: {e}")
    df = pd.DataFrame()

# 2. Hàm xử lý lọc dữ liệu
def filter_data(min_gpa):
    if df.empty:
        return pd.DataFrame()
    filtered_df = df[df['điểm gpa'] >= min_gpa]
    return filtered_df

# 3. CSS tùy chỉnh để vẽ chiếc "iPhone 17"
iphone_css = """
/* Ẩn phần footer mặc định của Gradio cho đẹp */
footer {display: none !important;}

/* Khung viền iPhone */
#iphone-frame {
    max-width: 400px !important;         /* Chiều rộng chuẩn của điện thoại */
    height: 800px !important;            /* Chiều dài màn hình */
    margin: 20px auto !important;        /* Căn giữa màn hình */
    border: 14px solid #1a1a1a !important; /* Viền đen dày của điện thoại */
    border-radius: 55px !important;      /* Bo góc tròn trịa */
    box-shadow: 0 20px 40px rgba(0,0,0,0.4), inset 0 0 10px rgba(0,0,0,0.1) !important; /* Đổ bóng 3D */
    position: relative !important;
    background: #f4f6f9 !important;      /* Màu nền màn hình (Hình nền) */
    overflow: hidden !important;
}

/* Dynamic Island (Đảo động) đặc trưng */
#dynamic-island {
    position: absolute;
    top: 15px;
    left: 50%;
    transform: translateX(-50%);
    width: 120px;
    height: 35px;
    background-color: #000000;
    border-radius: 20px;
    z-index: 1000;
    box-shadow: inset 0 0 5px rgba(255,255,255,0.2);
}

/* Khu vực chứa nội dung App (tránh bị Dynamic Island che mất) */
#app-content {
    height: 100%;
    overflow-y: auto;            /* Cho phép cuộn nếu nội dung dài */
    padding: 70px 20px 20px 20px; /* Đẩy nội dung xuống dưới Dynamic Island */
    box-sizing: border-box;
}

/* Làm đẹp bảng dữ liệu để vừa với màn hình nhỏ */
.table-wrap {
    font-size: 12px;
}
"""

# 4. Thiết kế giao diện (UI) bên trong chiếc iPhone
with gr.Blocks(css=iphone_css, theme=gr.themes.Default()) as app:
    # Bọc toàn bộ trong khung iPhone
    with gr.Column(elem_id="iphone-frame"):
        # Thêm Dynamic Island
        gr.HTML('<div id="dynamic-island"></div>')

        # Nội dung App trên màn hình
        with gr.Column(elem_id="app-content"):
            gr.Markdown("## 📱 Tra cứu GPA", elem_classes="text-center")
            gr.Markdown("Kéo thanh trượt để lọc dữ liệu sinh viên.")

            # Thanh trượt
            min_gpa_input = gr.Slider(
                minimum=0.0, maximum=4.0, step=0.1, value=0.0,
                label="GPA tối thiểu", interactive=True
            )

            # Bảng hiển thị
            output_table = gr.Dataframe(
                value=df,
                label="Danh sách sinh viên",
                interactive=False
            )

    # Sự kiện khi kéo thanh trượt -> tự động cập nhật bảng không cần nút bấm
    min_gpa_input.change(
        fn=filter_data,
        inputs=min_gpa_input,
        outputs=output_table
    )

# 5. Chạy App (hiển thị trực tiếp trong Colab)
app.launch(inline=True, share=True)

/tmp/ipykernel_18251/215414768.py:68: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=iphone_css, theme=gr.themes.Default()) as app:
/tmp/ipykernel_18251/215414768.py:68: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=iphone_css, theme=gr.themes.Default()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5f178bb60360d6ca0f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
import gradio as gr
import pandas as pd

# 1. Đọc và làm sạch dữ liệu
file_path = '/content/gpa.csv'
try:
    df = pd.read_csv(file_path, sep=';')
    df['điểm gpa'] = pd.to_numeric(df['điểm gpa'], errors='coerce')
    df = df.dropna(subset=['điểm gpa'])
except Exception as e:
    print(f"Lỗi đọc file: {e}")
    df = pd.DataFrame()

# 2. Hàm xử lý lọc dữ liệu
def filter_data(min_gpa):
    if df.empty:
        return pd.DataFrame()
    filtered_df = df[df['điểm gpa'] >= min_gpa]
    return filtered_df

# 3. CSS Sửa lỗi & Tối ưu
iphone_css = """
/* Ẩn footer Gradio */
footer {display: none !important;}

/* Khung viền iPhone chuẩn */
#iphone-frame {
    max-width: 400px !important;
    height: 800px !important;
    margin: 20px auto !important;
    border: 14px solid #1a1a1a !important;
    border-radius: 55px !important;
    box-shadow: 0 20px 40px rgba(0,0,0,0.4), inset 0 0 10px rgba(0,0,0,0.1) !important;
    background: #f4f6f9 !important;
    position: relative !important; /* Quan trọng để neo Dynamic Island */
    overflow: hidden !important;
    padding: 0px !important; /* Xóa padding mặc định của Gradio */
}

/* Khu vực nội dung App */
#app-content {
    height: 100%;
    overflow-y: auto !important;
    padding: 0px 20px 20px 20px !important;
}

/* Ẩn thanh cuộn cho đẹp (tuỳ chọn) */
#app-content::-webkit-scrollbar {
    display: none;
}
"""

# 4. Giao diện App
with gr.Blocks(css=iphone_css, theme=gr.themes.Default()) as app:

    with gr.Column(elem_id="iphone-frame"):

        # DYNAMIC ISLAND CỐ ĐỊNH TẠI ĐÂY (Dùng Inline CSS để không bị Gradio ghi đè)
        gr.HTML('''
        <div style="position: absolute; top: 12px; left: 50%; transform: translateX(-50%);
                    width: 120px; height: 35px; background-color: #000; border-radius: 20px;
                    z-index: 1000; display: flex; align-items: center; justify-content: flex-end; padding-right: 12px;
                    box-shadow: 0 2px 10px rgba(0,0,0,0.2);">
            <div style="width: 12px; height: 12px; background: #111; border-radius: 50%;
                        box-shadow: inset 0 0 4px rgba(255,255,255,0.3), inset 0 0 1px rgba(0,0,255,0.2);"></div>
        </div>
        <div style="height: 60px; width: 100%;"></div>
        ''')

        # Nội dung App
        with gr.Column(elem_id="app-content"):
            gr.Markdown("<h3 style='text-align: center; margin-bottom: 0;'>📱 Tra cứu GPA</h3>")
            gr.Markdown("<p style='text-align: center; font-size: 14px; color: gray;'>Kéo thanh trượt để lọc dữ liệu sinh viên</p>")

            # Thanh trượt
            min_gpa_input = gr.Slider(
                minimum=0.0, maximum=4.0, step=0.1, value=0.0,
                label="GPA tối thiểu", interactive=True
            )

            # Bảng hiển thị
            output_table = gr.Dataframe(
                value=df,
                label="Danh sách sinh viên",
                interactive=False,
                wrap=True
            )

    # Sự kiện lọc dữ liệu
    min_gpa_input.change(
        fn=filter_data,
        inputs=min_gpa_input,
        outputs=output_table
    )

# 5. Chạy App
app.launch(inline=True, share=True)

/tmp/ipykernel_18251/4182010656.py:54: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=iphone_css, theme=gr.themes.Default()) as app:
/tmp/ipykernel_18251/4182010656.py:54: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=iphone_css, theme=gr.themes.Default()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://da7e5be775624b9e8f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
import gradio as gr
import pandas as pd

# 1. Đọc và làm sạch dữ liệu
file_path = '/content/gpa.csv'
try:
    df = pd.read_csv(file_path, sep=';')
    df['điểm gpa'] = pd.to_numeric(df['điểm gpa'], errors='coerce')
    df = df.dropna(subset=['điểm gpa'])
except Exception as e:
    print(f"Lỗi đọc file: {e}")
    df = pd.DataFrame(columns=['điểm gpa', 'bạn dành bao nhiêu thời gian cho 1 tuần để học', 'bạn có làm thêm part time không', 'thời gian ngủ của bạn mỗi ngày'])

# 2. Hàm tạo giao diện "Bảng tin" (Feed) giống Instagram/Facebook
def generate_feed(min_gpa):
    if df.empty:
        return "<div style='text-align:center; padding: 20px; color: #888;'>Không có dữ liệu</div>"

    filtered_df = df[df['điểm gpa'] >= min_gpa].head(30) # Giới hạn 30 người để app chạy mượt

    if filtered_df.empty:
        return "<div style='text-align:center; padding: 40px; color: #888; font-size: 14px;'>Không tìm thấy sinh viên nào phù hợp 😢</div>"

    html = "<div style='padding-bottom: 80px;'>" # Chừa khoảng trống cho thanh menu dưới cùng

    for idx, row in filtered_df.iterrows():
        gpa = row['điểm gpa']
        study_time = row.get('bạn dành bao nhiêu thời gian cho 1 tuần để học', 'N/A')
        part_time = row.get('bạn có làm thêm part time không', 'N/A')
        sleep_time = row.get('thời gian ngủ của bạn mỗi ngày', 'N/A')

        # Thiết kế một "Card" (Bài viết) cho mỗi sinh viên
        html += f"""
        <div style="background: white; border-radius: 16px; padding: 16px; margin-bottom: 16px; box-shadow: 0 4px 12px rgba(0,0,0,0.05); border: 1px solid #f1f1f1;">
            <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 12px;">
                <div style="display: flex; align-items: center;">
                    <div style="width: 42px; height: 42px; border-radius: 50%; background: linear-gradient(45deg, #00529c, #007aff); color: white; display: flex; justify-content: center; align-items: center; font-weight: bold; font-size: 16px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                        SV
                    </div>
                    <div style="margin-left: 12px;">
                        <div style="font-weight: 700; font-size: 15px; color: #262626; letter-spacing: -0.3px;">Sinh viên #{idx+1}</div>
                        <div style="font-size: 12px; color: #8e8e8e;">Ho Chi Minh City, Vietnam</div>
                    </div>
                </div>
                <div style="background: #f0f7ff; color: #00529c; padding: 6px 12px; border-radius: 20px; font-weight: 800; font-size: 14px;">
                    ⭐ {gpa}
                </div>
            </div>

            <div style="font-size: 13.5px; color: #262626; line-height: 1.6;">
                <span style="display:inline-block; background: #f8f9fa; padding: 4px 10px; border-radius: 8px; margin: 0 6px 6px 0; border: 1px solid #eee;">📚 Tự học: <b>{study_time}h</b></span>
                <span style="display:inline-block; background: #f8f9fa; padding: 4px 10px; border-radius: 8px; margin: 0 6px 6px 0; border: 1px solid #eee;">💼 Part-time: <b>{part_time}</b></span>
                <span style="display:inline-block; background: #f8f9fa; padding: 4px 10px; border-radius: 8px; margin: 0 6px 6px 0; border: 1px solid #eee;">💤 Ngủ: <b>{sleep_time}h</b></span>
            </div>

            <div style="display: flex; gap: 16px; margin-top: 12px; padding-top: 12px; border-top: 1px solid #f1f1f1; color: #262626; font-size: 20px;">
                <span style="cursor: pointer;">♡</span>
                <span style="cursor: pointer;">💬</span>
                <span style="cursor: pointer;">↗</span>
            </div>
        </div>
        """
    html += "</div>"
    return html

# 3. CSS Thiết kế giao diện App chuẩn
iphone_css = """
footer {display: none !important;}

/* Khung điện thoại */
#iphone-frame {
    max-width: 400px !important;
    height: 800px !important;
    margin: 20px auto !important;
    border: 14px solid #000 !important;
    border-radius: 55px !important;
    box-shadow: 0 25px 50px rgba(0,0,0,0.5), inset 0 0 10px rgba(0,0,0,0.1) !important;
    background: #fafafa !important; /* Màu nền app sáng giống FB/IG */
    position: relative !important;
    overflow: hidden !important;
    padding: 0px !important;
}

/* Khu vực cuộn của bảng tin */
#app-content {
    height: 100%;
    overflow-y: auto !important;
    padding: 110px 16px 20px 16px !important; /* Đẩy xuống để chừa chỗ cho Header & Filter */
}

#app-content::-webkit-scrollbar { display: none; }

/* Thanh điều hướng dưới cùng (Bottom Nav) */
.bottom-nav {
    position: absolute;
    bottom: 0;
    left: 0;
    width: 100%;
    height: 70px;
    background: rgba(255, 255, 255, 0.95);
    backdrop-filter: blur(10px);
    border-top: 1px solid #e0e0e0;
    display: flex;
    justify-content: space-around;
    align-items: center;
    z-index: 1002;
    padding-bottom: 15px; /* Khu vực vuốt Home của iPhone */
}
.nav-icon {
    font-size: 24px;
    color: #262626;
}
"""

# 4. Xây dựng giao diện bằng Gradio
with gr.Blocks(css=iphone_css, theme=gr.themes.Base()) as app:

    with gr.Column(elem_id="iphone-frame"):

        # --- DYNAMIC ISLAND ---
        gr.HTML('''
        <div style="position: absolute; top: 12px; left: 50%; transform: translateX(-50%);
                    width: 120px; height: 35px; background-color: #000; border-radius: 20px;
                    z-index: 1005; display: flex; align-items: center; justify-content: flex-end; padding-right: 12px;">
            <div style="width: 12px; height: 12px; background: #111; border-radius: 50%;
                        box-shadow: inset 0 0 4px rgba(255,255,255,0.3);"></div>
        </div>
        ''')

        # --- HEADER (Thanh tiêu đề giống Instagram) ---
        gr.HTML('''
        <div style="position: absolute; top: 0; left: 0; width: 100%; height: 95px;
                    background: rgba(255, 255, 255, 0.95); backdrop-filter: blur(10px);
                    border-bottom: 1px solid #e0e0e0; z-index: 1001;
                    display: flex; align-items: flex-end; justify-content: space-between;
                    padding: 0 16px 12px 16px; box-sizing: border-box;">

            <div style="display: flex; align-items: center;">
                <img src="/content/Logo_UEH_xanh.png" style="height: 28px; object-fit: contain; margin-right: 8px;">
                <span style="font-size: 20px; font-weight: 700; color: #00529c; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;">Feed</span>
            </div>

            <div style="display: flex; gap: 16px; font-size: 24px; color: #262626;">
                <span>♡</span>
                <span>💬</span>
            </div>
        </div>
        ''')

        # --- BOTTOM NAV (Thanh điều hướng dưới cùng) ---
        gr.HTML('''
        <div class="bottom-nav">
            <span class="nav-icon">🏠</span>
            <span class="nav-icon">🔍</span>
            <span class="nav-icon">➕</span>
            <span class="nav-icon">🎬</span>
            <span class="nav-icon" style="width: 28px; height: 28px; border-radius: 50%; background: #ddd; display: inline-block;"></span>
        </div>
        ''')

        # --- CONTENT (Nội dung chính) ---
        with gr.Column(elem_id="app-content"):

            # Thanh trượt bộ lọc (được thiết kế ngầm giống thanh "Story" hoặc bộ lọc nhanh)
            gr.Markdown("<div style='font-size: 13px; font-weight: 600; margin-bottom: -10px; color: #262626;'>Lọc GPA tối thiểu:</div>")
            min_gpa_input = gr.Slider(
                minimum=0.0, maximum=4.0, step=0.1, value=0.0,
                show_label=False, interactive=True
            )

            # Vùng chứa Feed bài viết
            feed_output = gr.HTML(value=generate_feed(0.0))

    # Sự kiện cập nhật Bảng tin khi kéo thanh trượt
    min_gpa_input.change(
        fn=generate_feed,
        inputs=min_gpa_input,
        outputs=feed_output
    )

# 5. Khởi chạy
app.launch(inline=True, share=True)

/tmp/ipykernel_18251/4220372604.py:116: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=iphone_css, theme=gr.themes.Base()) as app:
/tmp/ipykernel_18251/4220372604.py:116: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=iphone_css, theme=gr.themes.Base()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3c194fc950ea8fafdf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [12]:
import gradio as gr
import pandas as pd

# 1. Đọc và làm sạch dữ liệu
file_path = '/content/gpa.csv'
try:
    df = pd.read_csv(file_path, sep=';')
    df['điểm gpa'] = pd.to_numeric(df['điểm gpa'], errors='coerce')
    df = df.dropna(subset=['điểm gpa'])
except Exception as e:
    print(f"Lỗi đọc file: {e}")
    df = pd.DataFrame(columns=['điểm gpa', 'bạn dành bao nhiêu thời gian cho 1 tuần để học', 'bạn có làm thêm part time không', 'thời gian ngủ của bạn mỗi ngày'])

# 2. Hàm tạo giao diện "Bảng tin" (Feed)
def generate_feed(min_gpa):
    if df.empty:
        return "<div style='text-align:center; padding: 20px; color: #888;'>Không có dữ liệu</div>"

    filtered_df = df[df['điểm gpa'] >= min_gpa].head(30)

    if filtered_df.empty:
        return "<div style='text-align:center; padding: 40px; color: #888; font-size: 14px;'>Không tìm thấy sinh viên nào phù hợp 😢</div>"

    html = "<div style='padding-bottom: 80px;'>"

    for idx, row in filtered_df.iterrows():
        gpa = row['điểm gpa']
        study_time = row.get('bạn dành bao nhiêu thời gian cho 1 tuần để học', 'N/A')
        part_time = row.get('bạn có làm thêm part time không', 'N/A')
        sleep_time = row.get('thời gian ngủ của bạn mỗi ngày', 'N/A')

        html += f"""
        <div style="background: white; border-radius: 16px; padding: 16px; margin-bottom: 16px; box-shadow: 0 4px 12px rgba(0,0,0,0.05); border: 1px solid #f1f1f1;">
            <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 12px;">
                <div style="display: flex; align-items: center;">
                    <div style="width: 42px; height: 42px; border-radius: 50%; background: linear-gradient(45deg, #00529c, #007aff); color: white; display: flex; justify-content: center; align-items: center; font-weight: bold; font-size: 16px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                        SV
                    </div>
                    <div style="margin-left: 12px;">
                        <div style="font-weight: 700; font-size: 15px; color: #262626; letter-spacing: -0.3px;">Sinh viên #{idx+1}</div>
                        <div style="font-size: 12px; color: #8e8e8e;">Ho Chi Minh City</div>
                    </div>
                </div>
                <div style="background: #f0f7ff; color: #00529c; padding: 6px 12px; border-radius: 20px; font-weight: 800; font-size: 14px;">
                    ⭐ {gpa}
                </div>
            </div>

            <div style="font-size: 13.5px; color: #262626; line-height: 1.6;">
                <span style="display:inline-block; background: #f8f9fa; padding: 4px 10px; border-radius: 8px; margin: 0 6px 6px 0; border: 1px solid #eee;">📚 Tự học: <b>{study_time}h</b></span>
                <span style="display:inline-block; background: #f8f9fa; padding: 4px 10px; border-radius: 8px; margin: 0 6px 6px 0; border: 1px solid #eee;">💼 Part-time: <b>{part_time}</b></span>
                <span style="display:inline-block; background: #f8f9fa; padding: 4px 10px; border-radius: 8px; margin: 0 6px 6px 0; border: 1px solid #eee;">💤 Ngủ: <b>{sleep_time}h</b></span>
            </div>

            <div style="display: flex; gap: 16px; margin-top: 12px; padding-top: 12px; border-top: 1px solid #f1f1f1; color: #262626; font-size: 20px;">
                <span style="cursor: pointer;">♡</span>
                <span style="cursor: pointer;">💬</span>
                <span style="cursor: pointer;">↗</span>
            </div>
        </div>
        """
    html += "</div>"
    return html

# 3. CSS Giao diện
iphone_css = """
footer {display: none !important;}

#iphone-frame {
    max-width: 400px !important;
    height: 800px !important;
    margin: 20px auto !important;
    border: 14px solid #000 !important;
    border-radius: 55px !important;
    box-shadow: 0 25px 50px rgba(0,0,0,0.5), inset 0 0 10px rgba(0,0,0,0.1) !important;
    background: #fafafa !important;
    position: relative !important;
    overflow: hidden !important;
    padding: 0px !important;
}

#app-content {
    height: 100%;
    overflow-y: auto !important;
    padding: 110px 16px 20px 16px !important;
}

#app-content::-webkit-scrollbar { display: none; }

.bottom-nav {
    position: absolute;
    bottom: 0;
    left: 0;
    width: 100%;
    height: 70px;
    background: rgba(255, 255, 255, 0.95);
    backdrop-filter: blur(10px);
    border-top: 1px solid #e0e0e0;
    display: flex;
    justify-content: space-around;
    align-items: center;
    z-index: 1002;
    padding-bottom: 15px;
}
.nav-icon { font-size: 24px; color: #262626; }
"""

# 4. Xây dựng giao diện bằng Gradio
with gr.Blocks(css=iphone_css, theme=gr.themes.Base()) as app:

    with gr.Column(elem_id="iphone-frame"):

        # --- DYNAMIC ISLAND ---
        gr.HTML('''
        <div style="position: absolute; top: 12px; left: 50%; transform: translateX(-50%);
                    width: 120px; height: 35px; background-color: #000; border-radius: 20px;
                    z-index: 1005; display: flex; align-items: center; justify-content: flex-end; padding-right: 12px;">
            <div style="width: 12px; height: 12px; background: #111; border-radius: 50%;
                        box-shadow: inset 0 0 4px rgba(255,255,255,0.3);"></div>
        </div>
        ''')

        # --- HEADER VỚI ẢNH CỦA BẠN ---
        gr.HTML('''
        <div style="position: absolute; top: 0; left: 0; width: 100%; height: 95px;
                    background: rgba(255, 255, 255, 0.95); backdrop-filter: blur(10px);
                    border-bottom: 1px solid #e0e0e0; z-index: 1001;
                    display: flex; align-items: flex-end; justify-content: space-between;
                    padding: 0 16px 12px 16px; box-sizing: border-box;">

            <div style="display: flex; align-items: center;">
                <img src="/content/Ảnh màn hình 2026-05-20 lúc 05.18.25.png"
                     style="height: 32px; width: 32px; border-radius: 8px; object-fit: cover; margin-right: 10px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <span style="font-size: 20px; font-weight: 700; color: #262626; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;">Feed</span>
            </div>

            <div style="display: flex; gap: 16px; font-size: 24px; color: #262626;">
                <span>♡</span>
                <span>💬</span>
            </div>
        </div>
        ''')

        # --- BOTTOM NAV ---
        gr.HTML('''
        <div class="bottom-nav">
            <span class="nav-icon">🏠</span>
            <span class="nav-icon">🔍</span>
            <span class="nav-icon">➕</span>
            <span class="nav-icon">🎬</span>
            <span class="nav-icon" style="width: 28px; height: 28px; border-radius: 50%; background: #ddd; display: inline-block;"></span>
        </div>
        ''')

        # --- CONTENT ---
        with gr.Column(elem_id="app-content"):
            gr.Markdown("<div style='font-size: 13px; font-weight: 600; margin-bottom: -10px; color: #262626;'>Lọc GPA tối thiểu:</div>")
            min_gpa_input = gr.Slider(
                minimum=0.0, maximum=4.0, step=0.1, value=0.0,
                show_label=False, interactive=True
            )
            feed_output = gr.HTML(value=generate_feed(0.0))

    min_gpa_input.change(fn=generate_feed, inputs=min_gpa_input, outputs=feed_output)

# 5. Khởi chạy
app.launch(inline=True, share=True)

/tmp/ipykernel_18251/3869027725.py:109: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=iphone_css, theme=gr.themes.Base()) as app:
/tmp/ipykernel_18251/3869027725.py:109: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=iphone_css, theme=gr.themes.Base()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b6cc788b8d855593ee.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [13]:
import gradio as gr
import pandas as pd

# 1. Đọc và làm sạch dữ liệu
file_path = '/content/gpa.csv'
try:
    df = pd.read_csv(file_path, sep=';')
    df['điểm gpa'] = pd.to_numeric(df['điểm gpa'], errors='coerce')
    df = df.dropna(subset=['điểm gpa'])
except Exception as e:
    print(f"Lỗi đọc file: {e}")
    df = pd.DataFrame(columns=['điểm gpa', 'bạn dành bao nhiêu thời gian cho 1 tuần để học', 'bạn có làm thêm part time không', 'thời gian ngủ của bạn mỗi ngày'])

# 2. Hàm tạo giao diện "Bảng tin" (Feed)
def generate_feed(min_gpa):
    if df.empty:
        return "<div style='text-align:center; padding: 20px; color: #888;'>Không có dữ liệu</div>"

    filtered_df = df[df['điểm gpa'] >= min_gpa].head(30)

    if filtered_df.empty:
        return "<div style='text-align:center; padding: 40px; color: #888; font-size: 14px;'>Không tìm thấy sinh viên nào phù hợp 😢</div>"

    html = "<div style='padding-bottom: 80px;'>"

    for idx, row in filtered_df.iterrows():
        gpa = row['điểm gpa']
        study_time = row.get('bạn dành bao nhiêu thời gian cho 1 tuần để học', 'N/A')
        part_time = row.get('bạn có làm thêm part time không', 'N/A')
        sleep_time = row.get('thời gian ngủ của bạn mỗi ngày', 'N/A')

        html += f"""
        <div style="background: white; border-radius: 16px; padding: 16px; margin-bottom: 16px; box-shadow: 0 4px 12px rgba(0,0,0,0.05); border: 1px solid #f1f1f1;">
            <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 12px;">
                <div style="display: flex; align-items: center;">
                    <div style="width: 42px; height: 42px; border-radius: 50%; background: linear-gradient(45deg, #00529c, #007aff); color: white; display: flex; justify-content: center; align-items: center; font-weight: bold; font-size: 16px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                        SV
                    </div>
                    <div style="margin-left: 12px;">
                        <div style="font-weight: 700; font-size: 15px; color: #262626; letter-spacing: -0.3px;">Sinh viên #{idx+1}</div>
                        <div style="font-size: 12px; color: #8e8e8e;">Ho Chi Minh City</div>
                    </div>
                </div>
                <div style="background: #f0f7ff; color: #00529c; padding: 6px 12px; border-radius: 20px; font-weight: 800; font-size: 14px;">
                    ⭐ {gpa}
                </div>
            </div>

            <div style="font-size: 13.5px; color: #262626; line-height: 1.6;">
                <span style="display:inline-block; background: #f8f9fa; padding: 4px 10px; border-radius: 8px; margin: 0 6px 6px 0; border: 1px solid #eee;">📚 Tự học: <b>{study_time}h</b></span>
                <span style="display:inline-block; background: #f8f9fa; padding: 4px 10px; border-radius: 8px; margin: 0 6px 6px 0; border: 1px solid #eee;">💼 Part-time: <b>{part_time}</b></span>
                <span style="display:inline-block; background: #f8f9fa; padding: 4px 10px; border-radius: 8px; margin: 0 6px 6px 0; border: 1px solid #eee;">💤 Ngủ: <b>{sleep_time}h</b></span>
            </div>

            <div style="display: flex; gap: 16px; margin-top: 12px; padding-top: 12px; border-top: 1px solid #f1f1f1; color: #262626; font-size: 20px;">
                <span style="cursor: pointer;">♡</span>
                <span style="cursor: pointer;">💬</span>
                <span style="cursor: pointer;">↗</span>
            </div>
        </div>
        """
    html += "</div>"
    return html

# 3. CSS Giao diện
iphone_css = """
footer {display: none !important;}

#iphone-frame {
    max-width: 400px !important;
    height: 800px !important;
    margin: 20px auto !important;
    border: 14px solid #000 !important;
    border-radius: 55px !important;
    box-shadow: 0 25px 50px rgba(0,0,0,0.5), inset 0 0 10px rgba(0,0,0,0.1) !important;
    background: #fafafa !important;
    position: relative !important;
    overflow: hidden !important;
    padding: 0px !important;
}

#app-content {
    height: 100%;
    overflow-y: auto !important;
    padding: 110px 16px 20px 16px !important;
}

#app-content::-webkit-scrollbar { display: none; }

.bottom-nav {
    position: absolute;
    bottom: 0;
    left: 0;
    width: 100%;
    height: 70px;
    background: rgba(255, 255, 255, 0.95);
    backdrop-filter: blur(10px);
    border-top: 1px solid #e0e0e0;
    display: flex;
    justify-content: space-around;
    align-items: center;
    z-index: 1002;
    padding-bottom: 15px;
}
.nav-icon { font-size: 24px; color: #262626; }
"""

# 4. Xây dựng giao diện bằng Gradio
with gr.Blocks(css=iphone_css, theme=gr.themes.Base()) as app:

    with gr.Column(elem_id="iphone-frame"):

        # --- DYNAMIC ISLAND ---
        gr.HTML('''
        <div style="position: absolute; top: 12px; left: 50%; transform: translateX(-50%);
                    width: 120px; height: 35px; background-color: #000; border-radius: 20px;
                    z-index: 1005; display: flex; align-items: center; justify-content: flex-end; padding-right: 12px;">
            <div style="width: 12px; height: 12px; background: #111; border-radius: 50%;
                        box-shadow: inset 0 0 4px rgba(255,255,255,0.3);"></div>
        </div>
        ''')

        # --- HEADER VỚI ẢNH CỦA BẠN ---
        gr.HTML('''
        <div style="position: absolute; top: 0; left: 0; width: 100%; height: 95px;
                    background: rgba(255, 255, 255, 0.95); backdrop-filter: blur(10px);
                    border-bottom: 1px solid #e0e0e0; z-index: 1001;
                    display: flex; align-items: flex-end; justify-content: space-between;
                    padding: 0 16px 12px 16px; box-sizing: border-box;">

            <div style="display: flex; align-items: center;">
                <img src="/content/Ảnh màn hình 2026-05-20 lúc 05.18.25.png"
                     style="height: 32px; width: 32px; border-radius: 8px; object-fit: cover; margin-right: 10px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <span style="font-size: 20px; font-weight: 700; color: #262626; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;">Feed</span>
            </div>

            <div style="display: flex; gap: 16px; font-size: 24px; color: #262626;">
                <span>♡</span>
                <span>💬</span>
            </div>
        </div>
        ''')

        # --- BOTTOM NAV ---
        gr.HTML('''
        <div class="bottom-nav">
            <span class="nav-icon">🏠</span>
            <span class="nav-icon">🔍</span>
            <span class="nav-icon">➕</span>
            <span class="nav-icon">🎬</span>
            <span class="nav-icon" style="width: 28px; height: 28px; border-radius: 50%; background: #ddd; display: inline-block;"></span>
        </div>
        ''')

        # --- CONTENT ---
        with gr.Column(elem_id="app-content"):
            gr.Markdown("<div style='font-size: 13px; font-weight: 600; margin-bottom: -10px; color: #262626;'>Lọc GPA tối thiểu:</div>")
            min_gpa_input = gr.Slider(
                minimum=0.0, maximum=4.0, step=0.1, value=0.0,
                show_label=False, interactive=True
            )
            feed_output = gr.HTML(value=generate_feed(0.0))

    min_gpa_input.change(fn=generate_feed, inputs=min_gpa_input, outputs=feed_output)

# 5. Khởi chạy
app.launch(inline=True, share=True)

/tmp/ipykernel_18251/3869027725.py:109: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=iphone_css, theme=gr.themes.Base()) as app:
/tmp/ipykernel_18251/3869027725.py:109: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=iphone_css, theme=gr.themes.Base()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://18f51b833ab8a6c876.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
import gradio as gr
import pandas as pd

# 1. Đọc và làm sạch dữ liệu
file_path = '/content/gpa.csv'
try:
    df = pd.read_csv(file_path, sep=';')
    df['điểm gpa'] = pd.to_numeric(df['điểm gpa'], errors='coerce')
    df = df.dropna(subset=['điểm gpa'])
except Exception as e:
    print(f"Lỗi đọc file: {e}")
    df = pd.DataFrame(columns=['điểm gpa', 'bạn dành bao nhiêu thời gian cho 1 tuần để học', 'bạn có làm thêm part time không', 'thời gian ngủ của bạn mỗi ngày'])

# 2. Hàm tạo giao diện "Bảng tin" (Feed)
def generate_feed(min_gpa):
    if df.empty:
        return "<div style='text-align:center; padding: 20px; color: #888;'>Không có dữ liệu</div>"

    filtered_df = df[df['điểm gpa'] >= min_gpa].head(30)

    if filtered_df.empty:
        return "<div style='text-align:center; padding: 40px; color: #888; font-size: 14px;'>Không tìm thấy sinh viên nào phù hợp 😢</div>"

    html = "<div style='padding-bottom: 80px;'>"

    for idx, row in filtered_df.iterrows():
        gpa = row['điểm gpa']
        study_time = row.get('bạn dành bao nhiêu thời gian cho 1 tuần để học', 'N/A')
        part_time = row.get('bạn có làm thêm part time không', 'N/A')
        sleep_time = row.get('thời gian ngủ của bạn mỗi ngày', 'N/A')

        html += f"""
        <div style="background: white; border-radius: 16px; padding: 16px; margin-bottom: 16px; box-shadow: 0 4px 12px rgba(0,0,0,0.05); border: 1px solid #f1f1f1;">
            <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 12px;">
                <div style="display: flex; align-items: center;">
                    <div style="width: 42px; height: 42px; border-radius: 50%; background: linear-gradient(45deg, #00529c, #007aff); color: white; display: flex; justify-content: center; align-items: center; font-weight: bold; font-size: 16px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                        SV
                    </div>
                    <div style="margin-left: 12px;">
                        <div style="font-weight: 700; font-size: 15px; color: #262626; letter-spacing: -0.3px;">Sinh viên #{idx+1}</div>
                        <div style="font-size: 12px; color: #8e8e8e;">Ho Chi Minh City</div>
                    </div>
                </div>
                <div style="background: #f0f7ff; color: #00529c; padding: 6px 12px; border-radius: 20px; font-weight: 800; font-size: 14px;">
                    ⭐ {gpa}
                </div>
            </div>

            <div style="font-size: 13.5px; color: #262626; line-height: 1.6;">
                <span style="display:inline-block; background: #f8f9fa; padding: 4px 10px; border-radius: 8px; margin: 0 6px 6px 0; border: 1px solid #eee;">📚 Tự học: <b>{study_time}h</b></span>
                <span style="display:inline-block; background: #f8f9fa; padding: 4px 10px; border-radius: 8px; margin: 0 6px 6px 0; border: 1px solid #eee;">💼 Part-time: <b>{part_time}</b></span>
                <span style="display:inline-block; background: #f8f9fa; padding: 4px 10px; border-radius: 8px; margin: 0 6px 6px 0; border: 1px solid #eee;">💤 Ngủ: <b>{sleep_time}h</b></span>
            </div>

            <div style="display: flex; gap: 16px; margin-top: 12px; padding-top: 12px; border-top: 1px solid #f1f1f1; color: #262626; font-size: 20px;">
                <span style="cursor: pointer;">♡</span>
                <span style="cursor: pointer;">💬</span>
                <span style="cursor: pointer;">↗</span>
            </div>
        </div>
        """
    html += "</div>"
    return html

# 3. CSS Giao diện
iphone_css = """
footer {display: none !important;}

#iphone-frame {
    max-width: 400px !important;
    height: 800px !important;
    margin: 20px auto !important;
    border: 14px solid #000 !important;
    border-radius: 55px !important;
    box-shadow: 0 25px 50px rgba(0,0,0,0.5), inset 0 0 10px rgba(0,0,0,0.1) !important;
    background: #fafafa !important;
    position: relative !important;
    overflow: hidden !important;
    padding: 0px !important;
}

#app-content {
    height: 100%;
    overflow-y: auto !important;
    padding: 110px 16px 20px 16px !important;
}

#app-content::-webkit-scrollbar { display: none; }

.bottom-nav {
    position: absolute;
    bottom: 0;
    left: 0;
    width: 100%;
    height: 70px;
    background: rgba(255, 255, 255, 0.95);
    backdrop-filter: blur(10px);
    border-top: 1px solid #e0e0e0;
    display: flex;
    justify-content: space-around;
    align-items: center;
    z-index: 1002;
    padding-bottom: 15px;
}
.nav-icon { font-size: 24px; color: #262626; }
"""

# 4. Xây dựng giao diện bằng Gradio
with gr.Blocks(css=iphone_css, theme=gr.themes.Base()) as app:

    with gr.Column(elem_id="iphone-frame"):

        # --- DYNAMIC ISLAND ---
        gr.HTML('''
        <div style="position: absolute; top: 12px; left: 50%; transform: translateX(-50%);
                    width: 120px; height: 35px; background-color: #000; border-radius: 20px;
                    z-index: 1005; display: flex; align-items: center; justify-content: flex-end; padding-right: 12px;">
            <div style="width: 12px; height: 12px; background: #111; border-radius: 50%;
                        box-shadow: inset 0 0 4px rgba(255,255,255,0.3);"></div>
        </div>
        ''')

        # --- HEADER VỚI ẢNH CỦA BẠN ---
        gr.HTML('''
        <div style="position: absolute; top: 0; left: 0; width: 100%; height: 95px;
                    background: rgba(255, 255, 255, 0.95); backdrop-filter: blur(10px);
                    border-bottom: 1px solid #e0e0e0; z-index: 1001;
                    display: flex; align-items: flex-end; justify-content: space-between;
                    padding: 0 16px 12px 16px; box-sizing: border-box;">

            <div style="display: flex; align-items: center;">
                <img src="/content/Ảnh màn hình 2026-05-20 lúc 05.18.25.png"
                     style="height: 32px; width: 32px; border-radius: 8px; object-fit: cover; margin-right: 10px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <span style="font-size: 20px; font-weight: 700; color: #262626; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;">Feed</span>
            </div>

            <div style="display: flex; gap: 16px; font-size: 24px; color: #262626;">
                <span>♡</span>
                <span>💬</span>
            </div>
        </div>
        ''')

        # --- BOTTOM NAV ---
        gr.HTML('''
        <div class="bottom-nav">
            <span class="nav-icon">🏠</span>
            <span class="nav-icon">🔍</span>
            <span class="nav-icon">➕</span>
            <span class="nav-icon">🎬</span>
            <span class="nav-icon" style="width: 28px; height: 28px; border-radius: 50%; background: #ddd; display: inline-block;"></span>
        </div>
        ''')

        # --- CONTENT ---
        with gr.Column(elem_id="app-content"):
            gr.Markdown("<div style='font-size: 13px; font-weight: 600; margin-bottom: -10px; color: #262626;'>Lọc GPA tối thiểu:</div>")
            min_gpa_input = gr.Slider(
                minimum=0.0, maximum=4.0, step=0.1, value=0.0,
                show_label=False, interactive=True
            )
            feed_output = gr.HTML(value=generate_feed(0.0))

    min_gpa_input.change(fn=generate_feed, inputs=min_gpa_input, outputs=feed_output)

# 5. Khởi chạy
app.launch(inline=True, share=True)

/tmp/ipykernel_18251/3869027725.py:109: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=iphone_css, theme=gr.themes.Base()) as app:
/tmp/ipykernel_18251/3869027725.py:109: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=iphone_css, theme=gr.themes.Base()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b6209da1aee397d72e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [15]:
import gradio as gr
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

# 1. Đọc và huấn luyện mô hình dự đoán từ dữ liệu thực tế
file_path = '/content/gpa.csv'
is_model_ready = False

try:
    df = pd.read_csv(file_path, sep=';')

    # Trích xuất và làm sạch các cột quan trọng
    df['gpa'] = pd.to_numeric(df['điểm gpa'], errors='coerce')
    df['study_time'] = pd.to_numeric(df['bạn dành bao nhiêu thời gian cho 1 tuần để học'], errors='coerce')
    df['sleep_time'] = pd.to_numeric(df['thời gian ngủ của bạn mỗi ngày'], errors='coerce')

    # Xử lý các cột phân loại (Có/Không)
    df['part_time'] = df['bạn có làm thêm part time không'].astype(str).str.strip().str.lower()
    df['club'] = df['bạn có tham gia câu lạc bộ không '].astype(str).str.strip().str.lower()

    # Chuyển đổi Có/Không thành 1 và 0 cho AI hiểu
    df['part_time_num'] = df['part_time'].apply(lambda x: 1 if x == 'có' else 0)
    df['club_num'] = df['club'].apply(lambda x: 1 if x == 'có' else 0)

    # Lọc bỏ các dòng bị thiếu dữ liệu
    df = df.dropna(subset=['gpa', 'study_time', 'sleep_time'])

    # 2. Xây dựng mô hình K-Nearest Neighbors (Tìm những sinh viên giống bạn nhất)
    X = df[['study_time', 'sleep_time', 'part_time_num', 'club_num']]
    y = df['gpa']

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Lấy trung bình điểm của 5 sinh viên có thói quen giống nhất
    knn = KNeighborsRegressor(n_neighbors=5)
    knn.fit(X_scaled, y)

    is_model_ready = True
except Exception as e:
    print(f"Lỗi khởi tạo mô hình: {e}")

# 3. Hàm xử lý logic khi người dùng bấm nút Dự đoán
def predict_gpa(study_time, sleep_time, part_time, club):
    if not is_model_ready:
        return "<div style='text-align:center; padding: 20px; color:red;'>Lỗi dữ liệu. Không thể dự đoán!</div>"

    # Chuyển đổi đầu vào của người dùng
    pt_num = 1 if part_time == 'Có' else 0
    club_num = 1 if club == 'Có' else 0

    # Chuẩn bị dữ liệu cho AI
    user_data = pd.DataFrame([[study_time, sleep_time, pt_num, club_num]],
                             columns=['study_time', 'sleep_time', 'part_time_num', 'club_num'])
    user_scaled = scaler.transform(user_data)

    # Tiến hành dự đoán
    pred_gpa = knn.predict(user_scaled)[0]
    pred_gpa = max(0.0, min(4.0, pred_gpa)) # Đảm bảo GPA không vượt quá 4.0

    # Giao diện thẻ kết quả (Màu sắc thay đổi theo điểm số)
    if pred_gpa >= 3.6:
        color = "#10b981" # Xanh lá (Xuất sắc)
        msg = "Tuyệt vời! Thói quen này giúp bạn lọt top Xuất sắc. 🚀"
    elif pred_gpa >= 3.0:
        color = "#3b82f6" # Xanh dương (Giỏi)
        msg = "Rất tốt! Duy trì phong độ này nhé. 👏"
    elif pred_gpa >= 2.5:
        color = "#f59e0b" # Cam (Khá)
        msg = "Khá ổn, nhưng hãy thử tăng thời gian tự học xem sao! 📚"
    else:
        color = "#ef4444" # Đỏ (Cần cải thiện)
        msg = "Cảnh báo! Bạn cần thay đổi thói quen học tập ngay. ⚠️"

    html = f"""
    <div style="background: white; border-radius: 20px; padding: 25px 20px; text-align: center;
                box-shadow: 0 10px 25px rgba(0,0,0,0.1); border-top: 6px solid {color};
                margin-top: 20px; animation: popup 0.5s ease-out;">
        <h3 style="margin: 0; color: #4b5563; font-size: 16px;">Dự đoán GPA của bạn</h3>
        <div style="font-size: 54px; font-weight: 800; color: {color}; margin: 15px 0;">{pred_gpa:.2f}</div>
        <div style="background: {color}20; color: {color}; padding: 10px; border-radius: 12px; font-size: 14px; font-weight: 600;">
            {msg}
        </div>
        <p style="color: #9ca3af; font-size: 11px; margin-top: 15px; margin-bottom: 0;">
            (Dựa trên dữ liệu của các sinh viên có thói quen tương đồng với bạn)
        </p>
    </div>
    """
    return html

# 4. CSS Định dạng giao diện App
iphone_css = """
footer {display: none !important;}
#iphone-frame {
    max-width: 400px !important; height: 800px !important; margin: 20px auto !important;
    border: 14px solid #000 !important; border-radius: 55px !important;
    box-shadow: 0 25px 50px rgba(0,0,0,0.5), inset 0 0 10px rgba(0,0,0,0.1) !important;
    background: #f3f4f6 !important; position: relative !important;
    overflow: hidden !important; padding: 0px !important;
}
#app-content {
    height: 100%; overflow-y: auto !important;
    padding: 105px 20px 20px 20px !important;
}
#app-content::-webkit-scrollbar { display: none; }
"""

# 5. Xây dựng giao diện bằng Gradio
with gr.Blocks(css=iphone_css, theme=gr.themes.Soft()) as app:
    with gr.Column(elem_id="iphone-frame"):

        # --- DYNAMIC ISLAND & HEADER ---
        gr.HTML('''
        <div style="position: absolute; top: 12px; left: 50%; transform: translateX(-50%);
                    width: 120px; height: 35px; background-color: #000; border-radius: 20px;
                    z-index: 1005; display: flex; align-items: center; justify-content: flex-end; padding-right: 12px;">
            <div style="width: 12px; height: 12px; background: #111; border-radius: 50%;
                        box-shadow: inset 0 0 4px rgba(255,255,255,0.3);"></div>
        </div>
        <div style="position: absolute; top: 0; left: 0; width: 100%; height: 95px;
                    background: rgba(255, 255, 255, 0.95); backdrop-filter: blur(10px);
                    border-bottom: 1px solid #e5e7eb; z-index: 1001;
                    display: flex; align-items: flex-end; padding: 0 20px 15px 20px; box-sizing: border-box;">
            <div style="display: flex; align-items: center;">
                <img src="/content/Ảnh màn hình 2026-05-20 lúc 05.18.25.png"
                     style="height: 36px; width: 36px; border-radius: 10px; object-fit: cover; margin-right: 12px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <div>
                    <div style="font-size: 18px; font-weight: 800; color: #111827; line-height: 1.2;">GPA Predictor</div>
                    <div style="font-size: 12px; color: #6b7280; font-weight: 500;">AI Powered Analysis</div>
                </div>
            </div>
        </div>
        ''')

        # --- NỘI DUNG FORM DỰ ĐOÁN ---
        with gr.Column(elem_id="app-content"):
            gr.Markdown("<p style='font-size: 14.5px; color: #4b5563; font-weight: 500; margin-bottom: 10px;'>Hãy nhập thói quen sinh hoạt của bạn để AI dự đoán kết quả học tập nhé!</p>")

            with gr.Column(scale=1, variant="panel"):
                study_input = gr.Slider(minimum=0, maximum=50, step=1, value=15, label="📚 Giờ tự học 1 tuần", interactive=True)
                sleep_input = gr.Slider(minimum=4, maximum=12, step=1, value=7, label="💤 Giờ ngủ 1 ngày", interactive=True)

                with gr.Row():
                    part_time_input = gr.Radio(choices=["Có", "Không"], value="Không", label="💼 Có làm part-time?", interactive=True)
                    club_input = gr.Radio(choices=["Có", "Không"], value="Có", label="🎯 Tham gia CLB?", interactive=True)

            predict_btn = gr.Button("🔮 Phân tích & Dự đoán", variant="primary", size="lg")

            # Khu vực hiển thị kết quả trả về
            result_output = gr.HTML()

    # Gắn sự kiện: Khi bấm nút sẽ gọi hàm dự đoán và trả HTML ra vùng result_output
    predict_btn.click(
        fn=predict_gpa,
        inputs=[study_input, sleep_input, part_time_input, club_input],
        outputs=result_output
    )

app.launch(inline=True, share=True)

/tmp/ipykernel_18251/2626417650.py:111: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=iphone_css, theme=gr.themes.Soft()) as app:
/tmp/ipykernel_18251/2626417650.py:111: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=iphone_css, theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f74c6c8012b28b1027.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [17]:
import gradio as gr
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

# 1. Đọc và huấn luyện mô hình (giữ nguyên logic AI)
file_path = '/content/gpa.csv'
is_model_ready = False

try:
    df = pd.read_csv(file_path, sep=';')
    df['gpa'] = pd.to_numeric(df['điểm gpa'], errors='coerce')
    df['study_time'] = pd.to_numeric(df['bạn dành bao nhiêu thời gian cho 1 tuần để học'], errors='coerce')
    df['sleep_time'] = pd.to_numeric(df['thời gian ngủ của bạn mỗi ngày'], errors='coerce')
    df['part_time_num'] = df['bạn có làm thêm part time không'].apply(lambda x: 1 if str(x).lower() == 'có' else 0)
    df['club_num'] = df['bạn có tham gia câu lạc bộ không '].apply(lambda x: 1 if str(x).lower() == 'có' else 0)

    df = df.dropna(subset=['gpa', 'study_time', 'sleep_time'])
    X = df[['study_time', 'sleep_time', 'part_time_num', 'club_num']]
    y = df['gpa']
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    knn = KNeighborsRegressor(n_neighbors=5)
    knn.fit(X_scaled, y)
    is_model_ready = True
except Exception as e:
    print(f"Lỗi dữ liệu: {e}")

# 2. Hàm dự đoán
def predict_gpa(study_time, sleep_time, part_time, club):
    if not is_model_ready: return "Lỗi hệ thống."
    pt_num = 1 if part_time == 'Có' else 0
    club_num = 1 if club == 'Có' else 0
    user_data = pd.DataFrame([[study_time, sleep_time, pt_num, club_num]], columns=['study_time', 'sleep_time', 'part_time_num', 'club_num'])
    pred_gpa = knn.predict(scaler.transform(user_data))[0]
    pred_gpa = max(0.0, min(4.0, pred_gpa))

    return f"""
    <div style="text-align: center; padding: 20px;">
        <div style="font-size: 14px; color: #666;">GPA dự đoán của bạn là:</div>
        <div style="font-size: 48px; font-weight: 800; color: #00529c; margin: 10px 0;">{pred_gpa:.2f}</div>
        <div style="font-size: 14px; color: #333; font-weight: 500;">Dựa trên dữ liệu khảo sát sinh viên UEH</div>
    </div>
    """

# 3. CSS cho khung hình chữ nhật dọc (Web App style)
clean_css = """
#main-container {
    max-width: 450px !important;
    margin: 40px auto !important;
    border: 1px solid #e0e0e0 !important;
    border-radius: 20px !important;
    padding: 30px !important;
    background: #ffffff !important;
    box-shadow: 0 10px 30px rgba(0,0,0,0.05) !important;
}
"""

# 4. Giao diện
with gr.Blocks(css=clean_css, theme=gr.themes.Soft()) as app:
    with gr.Column(elem_id="main-container"):
        # Header với Logo
        gr.HTML(f'''
        <div style="display: flex; align-items: center; margin-bottom: 25px;">
            <img src="/content/Logo_UEH_xanh.png" style="width: 50px; height: 50px; border-radius: 12px; margin-right: 15px;">
            <div>
                <h2 style="margin:0; color: #333;">UEH GPA Predictor</h2>
                <span style="color: #888; font-size: 13px;">Công cụ phân tích thói quen học tập</span>
            </div>
        </div>
        ''')

        # Form nhập liệu
        study_input = gr.Slider(0, 50, 15, label="📚 Giờ tự học/tuần")
        sleep_input = gr.Slider(4, 12, 7, label="💤 Giờ ngủ/ngày")
        with gr.Row():
            part_time_input = gr.Radio(["Có", "Không"], label="💼 Làm Part-time?", value="Không")
            club_input = gr.Radio(["Có", "Không"], label="🎯 Tham gia CLB?", value="Có")

        predict_btn = gr.Button("🔮 Phân tích ngay", variant="primary")
        result_output = gr.HTML()

        predict_btn.click(predict_gpa, [study_input, sleep_input, part_time_input, club_input], result_output)

app.launch(inline=True, share=True)

/tmp/ipykernel_18251/53128686.py:61: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=clean_css, theme=gr.themes.Soft()) as app:
/tmp/ipykernel_18251/53128686.py:61: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=clean_css, theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://961ed5700da12af4e8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [18]:
import gradio as gr
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

# 1. Đọc và huấn luyện mô hình (giữ nguyên logic AI)
file_path = '/content/gpa.csv'
is_model_ready = False

try:
    df = pd.read_csv(file_path, sep=';')
    df['gpa'] = pd.to_numeric(df['điểm gpa'], errors='coerce')
    df['study_time'] = pd.to_numeric(df['bạn dành bao nhiêu thời gian cho 1 tuần để học'], errors='coerce')
    df['sleep_time'] = pd.to_numeric(df['thời gian ngủ của bạn mỗi ngày'], errors='coerce')
    df['part_time_num'] = df['bạn có làm thêm part time không'].apply(lambda x: 1 if str(x).lower() == 'có' else 0)
    df['club_num'] = df['bạn có tham gia câu lạc bộ không '].apply(lambda x: 1 if str(x).lower() == 'có' else 0)

    df = df.dropna(subset=['gpa', 'study_time', 'sleep_time'])
    X = df[['study_time', 'sleep_time', 'part_time_num', 'club_num']]
    y = df['gpa']
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    knn = KNeighborsRegressor(n_neighbors=5)
    knn.fit(X_scaled, y)
    is_model_ready = True
except Exception as e:
    print(f"Lỗi dữ liệu: {e}")

# 2. Hàm dự đoán
def predict_gpa(study_time, sleep_time, part_time, club):
    if not is_model_ready: return "<div style='color:red;'>Dữ liệu chưa sẵn sàng.</div>"
    pt_num = 1 if part_time == 'Có' else 0
    club_num = 1 if club == 'Có' else 0
    user_data = pd.DataFrame([[study_time, sleep_time, pt_num, club_num]], columns=['study_time', 'sleep_time', 'part_time_num', 'club_num'])
    pred_gpa = knn.predict(scaler.transform(user_data))[0]
    pred_gpa = max(0.0, min(4.0, pred_gpa))

    return f"""
    <div style="text-align: center; padding: 20px; background: #f8f9fa; border-radius: 15px; margin-top: 20px;">
        <div style="font-size: 14px; color: #666;">GPA dự đoán của bạn là:</div>
        <div style="font-size: 48px; font-weight: 800; color: #00529c; margin: 10px 0;">{pred_gpa:.2f}</div>
        <div style="font-size: 14px; color: #333; font-weight: 500;">Dựa trên dữ liệu khảo sát sinh viên UEH</div>
    </div>
    """

# 3. CSS cho giao diện Web App hiện đại
clean_css = """
#main-container {
    max-width: 450px !important;
    margin: 40px auto !important;
    border: 1px solid #e0e0e0 !important;
    border-radius: 20px !important;
    padding: 30px !important;
    background: #ffffff !important;
    box-shadow: 0 10px 30px rgba(0,0,0,0.05) !important;
}
"""

# 4. Giao diện App
with gr.Blocks(css=clean_css, theme=gr.themes.Soft()) as app:
    with gr.Column(elem_id="main-container"):
        # Header với Logo UEH mới
        gr.HTML(f'''
        <div style="display: flex; align-items: center; margin-bottom: 25px;">
            <img src="/content/Logo_UEH_xanh.png" style="width: 50px; height: 50px; object-fit: contain; margin-right: 15px;">
            <div>
                <h2 style="margin:0; color: #00529c;">UEH Predictor</h2>
                <span style="color: #888; font-size: 13px;">Công cụ dự đoán GPA bằng AI</span>
            </div>
        </div>
        ''')

        # Form nhập liệu
        study_input = gr.Slider(0, 50, 15, label="📚 Giờ tự học/tuần")
        sleep_input = gr.Slider(4, 12, 7, label="💤 Giờ ngủ/ngày")
        with gr.Row():
            part_time_input = gr.Radio(["Có", "Không"], label="💼 Làm Part-time?", value="Không")
            club_input = gr.Radio(["Có", "Không"], label="🎯 Tham gia CLB?", value="Có")

        predict_btn = gr.Button("🔮 Phân tích ngay", variant="primary")
        result_output = gr.HTML()

        predict_btn.click(predict_gpa, [study_input, sleep_input, part_time_input, club_input], result_output)

app.launch(inline=True, share=True)

/tmp/ipykernel_18251/3325858911.py:61: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=clean_css, theme=gr.themes.Soft()) as app:
/tmp/ipykernel_18251/3325858911.py:61: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=clean_css, theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dd87cda825ce4cce73.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
